In [4]:
# Import the libraries and read the data

# Data manipulation
import numpy as np
import pandas as pd

# For statistical analysis
from statsmodels.tsa.arima.model import ARIMA

# Import matplotlib and set the style
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

# Import and filter warnings
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings('ignore', 'statsmodels.tsa.arima.model.ARIMA',
                        FutureWarning)

import yfinance as yf
import pandas as pd

# Download AUD/USD daily data
data = yf.download("AUDUSD=X",
                   start="2020-01-01",
                   end="2021-01-01",
                   interval="1d",auto_adjust=True,multi_level_index=False)

# Save to CSV
data.to_csv("AUDUSD_2020.csv")

print(data.head())

# Drop the missing values
data = data.dropna()

[*********************100%***********************]  1 of 1 completed

               Close      High       Low      Open  Volume
Date                                                      
2020-01-01  0.701700  0.704500  0.701508  0.701508       0
2020-01-02  0.701951  0.702100  0.698130  0.701892       0
2020-01-03  0.698519  0.698661  0.693140  0.698370       0
2020-01-06  0.694420  0.695900  0.692602  0.694729       0
2020-01-07  0.693731  0.694100  0.686111  0.693720       0


In [7]:
# Setting the rolling window
rolling_window = int(len(data) * 0.70)

# Function to select the best model and forecast the price
def select_model_and_forecast(price_df):

    # The p,d and q values
    aic = []
    p = range(1, 3)
    d = range(0, 2)
    q = range(1, 3)

    # Empty list to store p,q,dist
    p_q_d = []

    # Get the aic score for different values of p, d and q
    for i in p:
        for j in d:
            for k in q:
                # Try to fit the model
                try:
                    gm = ARIMA(price_df, order=(i, j, k))
                    gm_fit = gm.fit()

                    # Calculate the AIC score
                    aic_temp = gm_fit.aic
                    keys_temp = (i, j, k)

                    # Save the AIC score and the p, d and q values
                    p_q_d.append(keys_temp)
                    aic.append(aic_temp)
                except:
                    # Do nothing when the model does not fit
                    print("Could not fit the model for: p,d,q =", i,j,k)
                    pass

    # Store values in dictionary
    aic_dict = {'p_q_d': p_q_d, 'aic': aic}

    # Create DataFrame from dictionary
    df = pd.DataFrame(aic_dict)

    # Get the minimum AIC value with the p,d and q values
    df = df[df['aic'] == df['aic'].min()].reset_index()

    # Only predict the price if the ARIMA model could be fit
    if len(df) > 0:
        # Fitting the optimum model
        gm = ARIMA(price_df, order=df.p_q_d[0])
        gm_fit = gm.fit()

        # making the forecast
        predicted_price = gm_fit.forecast()
        return predicted_price
    else:
        # Return -1 if the model could not be fit
        return -1

In [9]:
# Applying the `select_model_and_forecast` function to the price data
data['predicted'] = data['Close'].rolling(rolling_window).apply(select_model_and_forecast)

In [10]:
# Generate the signal
data['signal'] = np.where(data['predicted'] > data['Close'], 1, -1)

# Calculate strategy returns
data['strategy_returns'] = data['Close'].pct_change() * data['signal'].shift(1)
# Drop the rows where there is NaN value
data.dropna(inplace=True)